# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/titlyzaman25/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [15]:
!pip -q install -U datasets huggingface_hub

In [16]:
from huggingface_hub import whoami

info = whoami(token=HF_TOKEN)

print("Hugging Face user:", info["name"])

Hugging Face user: titlyzaman25


In [17]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [18]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print(march_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [19]:
import pandas as pd

march_df = pd.read_parquet(march_file)

print("Rows:", len(march_df))
print("Columns:", march_df.columns.tolist())

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [20]:
# Query 1 — Verify the grain

grain_check = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
)

print("Total rows:", len(march_df))
print("Unique date-client-content combinations:", len(grain_check))
print("Maximum rows per combination:", grain_check.max())
print("Duplicate combinations:", (grain_check > 1).sum())

Total rows: 9841378
Unique date-client-content combinations: 9841378
Maximum rows per combination: 1
Duplicate combinations: 0


In [21]:
# Query 2 — March 2026 row count and date span

print("March 2026 row count:", len(march_df))
print("Earliest report date:", march_df["report_date"].min())
print("Latest report date:", march_df["report_date"].max())

March 2026 row count: 9841378
Earliest report date: 2026-03-01
Latest report date: 2026-03-31


In [22]:
# Query 3 — GSC availability

gsc_available = march_df[
    march_df["gsc_data_available"].eq(True)
]
march_df["gsc_data_available"].eq(True)

print("Rows with GSC data available:", len(gsc_available))

Rows with GSC data available: 3611061


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
Unit of analysis: One row represents one content item for one client on one reporting date.

Time window: I will use the March 2026 partition as my development window. I will use this mid-panel month rather than the _sample table because the _sample table contains the final month, June 2026, and should be treated as a sealed test month.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Features:
gsc_impressions
gsc_clicks
gsc_avg_position
ga4_sessions
ga4_total_engagement_sec
Label/proxy: I will use a review-priority proxy for the ranking task. The proxy will represent an observed condition that can be used to prioritize content for human review rather than claiming that a page definitely needs a refresh.

Context
report_date
client_hash_id
content_hash_id

I will exclude client_hash_id and content_hash_id as ML features because they are identifiers used for grouping and joining, not meaningful performance signals. I will also exclude future outcome information and label-derived fields because they would not be available at the decision moment and could cause leakage.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1 — Verify the grain

grain_check = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
)

duplicates = grain_check[grain_check > 1]

print("Duplicate grain combinations:", len(duplicates))
print(duplicates.head())

# Query 2 — Row count and date range

print("Row count:", len(march_df))
print("Minimum report date:", march_df["report_date"].min())
print("Maximum report date:", march_df["report_date"].max())

Duplicate grain combinations: 0
Series([], dtype: int64)
Row count: 9841378
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31


In [26]:
# Query 3 — Verify data availability / missingness

required_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_total_engagement_sec"
]

for col in required_fields:
    missing = march_df[col].isna().sum()
    available = len(march_df) - missing
    pct_available = available / len(march_df) * 100

    print(
        f"{col}: "
        f"available={available:,}, "
        f"missing={missing:,}, "
        f"availability={pct_available:.2f}%"
    )

gsc_impressions: available=9,841,378, missing=0, availability=100.00%
gsc_clicks: available=9,841,378, missing=0, availability=100.00%
gsc_avg_position: available=3,611,061, missing=6,230,317, availability=36.69%
ga4_sessions: available=6,822,637, missing=3,018,741, availability=69.33%
ga4_total_engagement_sec: available=6,822,637, missing=3,018,741, availability=69.33%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
Data limitation: Data availability is uneven across the March 2026 partition. GSC impressions and clicks are available for all 9,841,378 rows, but average position is available for only 36.69% of rows. GA4 sessions and total engagement are available for 69.33% of rows. Therefore, missing data may affect comparisons across content, and the results should not be interpreted as complete or causal.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.